# Lição 11 - Protocolo Agente-para-Agente (A2A)


## Configuração


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## O que é o Protocolo A2A?

O **protocolo Agente-para-Agente (A2A)** é um padrão aberto que permite que agentes de IA se comuniquem,
descubram uns aos outros e colaborem — mesmo quando são construídos em diferentes frameworks ou hospedados
por serviços distintos.

Key concepts:

- **Descoberta** – Agentes publicam um *Cartão de Agente* que descreve suas capacidades, facilitando que outros agentes (ou orquestradores) encontrem o especialista certo para uma tarefa.
- **Troca de Mensagens** – Agentes trocam mensagens estruturadas por meio de um protocolo comum, de modo que uma solicitação de um agente possa ser entendida e atendida por outro independentemente de sua implementação interna.
- **Ciclo de Vida da Tarefa** – A2A defines states such as *submitted*, *working*, *completed*, and *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## Criando Agentes de Viagem Especializados


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Colaboração Multiagente via Fluxo de Trabalho

Conectamos os três agentes em um fluxo de trabalho sequencial que espelha a troca de mensagens A2A:

1. **CurrencyExchangeAgent** recebe a solicitação do usuário e fornece orientações sobre câmbio.
2. **ActivityPlannerAgent** recebe o contexto enriquecido e adiciona recomendações de atividades.
3. **TravelManagerAgent** sintetiza ambas as entradas em um resumo de viagem final.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Entendendo o A2A em Produção

Em um ambiente de produção, o protocolo A2A desbloqueia cenários poderosos entre serviços:

| Capacidade | Descrição |
|---|---|
| **Interoperabilidade entre frameworks** | Um agente construído com um framework pode delegar tarefas a um agente construído com qualquer outro framework compatível com A2A, permitindo verdadeira interoperabilidade entre organizações. |
| **Limites de serviço** | Agentes podem residir em microsserviços separados, em regiões de nuvem ou até mesmo em organizações diferentes, enquanto continuam colaborando de forma transparente. |
| **Descoberta dinâmica** | Um orquestrador pode consultar um registro de Agent Card em tempo de execução para encontrar o especialista mais adequado para uma determinada subtarefa. |
| **Streaming e notificações push** | O A2A oferece suporte a Server-Sent Events (SSE) para atualizações de progresso em tempo real e notificações push para tarefas de longa duração. |

O fluxo de trabalho que construímos acima é uma versão simplificada, em processo, desse padrão. Em uma implantação real
cada agente exporia um endpoint HTTP, publicaria um Agent Card e se comunicaria
via o protocolo A2A JSON-RPC.


## Resumo

Nesta lição você aprendeu:

1. **O que é o protocolo A2A** — um padrão aberto para descoberta entre agentes, mensagens,
   e gerenciamento de tarefas.
2. **Como criar agentes especializados** — um agente de Câmbio de Moedas, um agente Planejador de Atividades,
   e um orquestrador Gerenciador de Viagens.
3. **Como conectar agentes em um fluxo de trabalho** — usando `WorkflowBuilder` para modelar a passagem
   sequencial de mensagens entre agentes.
4. **Como o A2A funciona em produção** — permitindo colaboração entre diferentes frameworks e serviços
   com descoberta dinâmica e atualizações em streaming.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Isenção de responsabilidade:
Este documento foi traduzido utilizando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos empenhemos em manter a precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional realizada por um tradutor humano. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações equivocadas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
